# Football RAG — Full Pipeline (Merged & Improved)

**Notebook structure:**
1. Install & Imports
2. Global Config (single source of truth)
3. Data Fetching & Preprocessing
4. Chunking — Fixed / Recursive / Semantic *(with fixes)*
5. Chunk Quality Report
6. Embed & Upsert to Pinecone
7. Pinecone Sanity Check
8. BM25 Index Build
9. Retrieval Helpers (Dense · BM25 · RRF · Hybrid · CrossEncoder · **MMR**)
10. Curated Gold Evaluation Set *(hand-written + auto-generated)*
11. Retrieval Experiments — All 6 Configurations
12. Winner Selection
13. QA Generation (Groq)
14. LLM-as-Judge — Faithfulness & Response Relevancy *(per-category)*
15. Ablation Study — Chunking × Retrieval
16. Save Outputs




## 1. Install Dependencies

In [ ]:
!pip -q install wikipedia langchain langchain-text-splitters \
               sentence-transformers pinecone rank-bm25 \
               pandas numpy tqdm transformers accelerate openai
print("All packages installed.")


## 2. Global Config

Edit **only this cell** to tune the pipeline.


In [ ]:
import os, json, re, time, random, unicodedata
from datetime import date
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import wikipedia
from tqdm import tqdm
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pinecone import Pinecone, ServerlessSpec

wikipedia.set_lang("en")

# ── Runtime ────────────────────────────────────────────────────────────────────
DEV_MODE = True          # True → smaller eval sizes for fast iteration

# ── Pinecone ───────────────────────────────────────────────────────────────────
PINECONE_API_KEY = ""    # or set via env / Kaggle Secret
INDEX_NAME       = "sports-rag-v2"
EMBEDDING_MODEL  = "BAAI/bge-small-en-v1.5"   # dim=384
BATCH_SIZE       = 100

# ── Chunking ───────────────────────────────────────────────────────────────────
CHUNK_SIZE            = 1000   # chars
CHUNK_OVERLAP_FIXED   = 150
CHUNK_OVERLAP_REC     = 100
MIN_CHUNK_CHARS       = 80     # IMPROVED: was 50; filters more orphan fragments
SEM_MAX_CHARS         = 1000   # hard ceiling enforced by fixed override
SEM_MIN_CHARS         = 250
SEM_SIM_BREAK         = 0.32

# ── Retrieval ──────────────────────────────────────────────────────────────────
TOP_K                   = 10
RRF_K                   = 60
DENSE_TOP_K             = 30   # IMPROVED: was 50; reduces RRF noise
RERANK_CANDIDATES_TOP_K = 30
QA_CONTEXT_TOP_K        = 5
MMR_LAMBDA              = 0.6  # NEW: MMR diversity weight (0=diverse, 1=relevant)
CROSS_ENCODER_MODEL     = "BAAI/bge-reranker-base"
EMBED_MODEL_NAME        = EMBEDDING_MODEL

# ── LLM / Judge ────────────────────────────────────────────────────────────────
JUDGE_PROVIDER   = "groq"
JUDGE_MODEL_NAME = "llama-3.1-8b-instant"
QA_GEN_MODEL     = "llama-3.1-8b-instant"

# ── Eval sizing ────────────────────────────────────────────────────────────────
TARGET_EVAL_QUESTIONS  = 100
UNANSWERABLE_QUESTIONS = 15
QA_EVAL_SIZE           = 40 if DEV_MODE else TARGET_EVAL_QUESTIONS
JUDGE_EVAL_SIZE        = 10 if DEV_MODE else 30
ABLATION_EVAL_SIZE     = 8  if DEV_MODE else 20

# ── Load secrets (Kaggle) ──────────────────────────────────────────────────────
# Load from Kaggle Secrets if available, then allow environment variable override
try:
    from kaggle_secrets import UserSecretsClient
    _ks = UserSecretsClient()
    
    # Load PINECONE_API_KEY from Kaggle Secrets
    _pinecone_secret = _ks.get_secret("PINECONE_API_KEY")
    if _pinecone_secret:
        PINECONE_API_KEY = _pinecone_secret
        os.environ["PINECONE_API_KEY"] = _pinecone_secret
        print("✓ Loaded PINECONE_API_KEY from Kaggle Secrets.")
    
    # Load other API keys
    for _key in ["GROQ_API_KEY", "OPENAI_API_KEY"]:
        _val = _ks.get_secret(_key)
        if _val:
            os.environ[_key] = _val
            print(f"✓ Loaded {_key} from Kaggle Secrets.")
except Exception as _e:
    print(f"Kaggle Secrets not available (local environment): {_e}")
    pass

# Allow environment variable to override Kaggle Secret (for local dev)
if not PINECONE_API_KEY and os.environ.get("PINECONE_API_KEY"):
    PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY")
    print("✓ Loaded PINECONE_API_KEY from environment variable.")

FETCHED_DATE = str(date.today())
print(f"Config ready. DEV_MODE={DEV_MODE}  FETCHED_DATE={FETCHED_DATE}")


## 3. Data Fetching & Preprocessing

In [ ]:
TOPICS = [
    # Tournaments & competitions
    "FIFA World Cup", "2022 FIFA World Cup", "2018 FIFA World Cup", "2014 FIFA World Cup",
    "UEFA Champions League", "UEFA Europa League", "UEFA Conference League", "UEFA Super Cup",
    "UEFA European Championship", "Copa America", "Copa Libertadores", "Copa del Rey",
    "FA Cup", "EFL Cup", "Premier League", "La Liga", "Serie A", "Bundesliga", "Ligue 1",
    "Primeira Liga", "Eredivisie", "Major League Soccer", "Saudi Pro League",
    "AFC Champions League",
    # Clubs
    "Real Madrid CF", "FC Barcelona", "Manchester United F.C.", "Manchester City F.C.",
    "Liverpool F.C.", "Chelsea F.C.", "Arsenal F.C.", "Tottenham Hotspur F.C.",
    "Bayern Munich", "Borussia Dortmund", "Juventus FC", "Inter Milan", "A.C. Milan",
    "Napoli", "Paris Saint-Germain F.C.", "Atletico Madrid", "Ajax Amsterdam",
    "Benfica", "FC Porto",
    # Players
    "Lionel Messi", "Cristiano Ronaldo", "Neymar", "Kylian Mbappe", "Erling Haaland",
    "Kevin De Bruyne", "Luka Modric", "Karim Benzema", "Mohamed Salah",
    "Robert Lewandowski", "Virgil van Dijk", "Jude Bellingham", "Vinicius Junior",
    "Harry Kane", "Zinedine Zidane", "Ronaldinho", "Ronaldo (Brazilian footballer)",
    "Pele", "Diego Maradona", "Johan Cruyff",
    # National teams & history
    "Brazil national football team", "Argentina national football team",
    "France national football team", "Germany national football team",
    "Spain national football team", "England national football team",
    "Portugal national football team", "Italy national football team",
    "Netherlands national football team", "Croatia national football team",
    "History of association football", "Laws of the Game (association football)",
    "VAR (association football)", "FIFA", "UEFA", "Ballon d'Or",
    "FIFA The Best Men's Player",
]
print(f"Total topics: {len(TOPICS)}")


In [ ]:
def clean_text(text: str) -> str:
    text = re.sub(r"=={1,3}.*?=={1,3}", "", text)
    text = re.sub(r"\[\d+\]", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def preprocess_doc(doc: dict) -> dict:
    """Light-touch cleanup: remove IPA, hatnotes, numeric-only lines."""
    text  = doc["content"]
    title = doc["title"]
    # Remove IPA / phonetic parentheticals
    text = re.sub(r'\([^)]*[\u0080-\uFFFF][^)]*\)', ' ', text)
    text = re.sub(r'\[[^\]]*[\u0080-\uFFFF][^\]]*\]', ' ', text)
    # Strip Wikipedia hatnote boilerplate
    for pat in [
        r'^This article (is about|covers|describes)\b.*$',
        r'^For (other uses|the [\w ]+),? see\b.*$',
        r'^Not to be confused with\b.*$',
        r'^See also\s*:.*$',
        r'^Further information\s*:.*$',
        r'^This (section|article) (needs|requires|may need)\b.*$',
    ]:
        text = re.sub(pat, '', text, flags=re.MULTILINE | re.IGNORECASE)
    # Drop purely-numeric lines (stat table artefacts)
    text = re.sub(r'^\s*[\d,\.\s]+\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text  = unicodedata.normalize("NFC", text.strip())
    title = unicodedata.normalize("NFC", title)
    return {**doc, "content": text, "title": title}


raw_docs = []
failed   = []

for topic in tqdm(TOPICS, desc="Fetching Wikipedia"):
    try:
        page    = wikipedia.page(topic, auto_suggest=False)
        content = clean_text(page.content)
        if len(content) > 300:
            raw_docs.append({
                "title"       : page.title,
                "url"         : page.url,
                "content"     : content,
                "fetched_date": FETCHED_DATE,
            })
    except wikipedia.exceptions.DisambiguationError as e:
        try:
            page    = wikipedia.page(e.options[0], auto_suggest=False)
            content = clean_text(page.content)
            raw_docs.append({
                "title": page.title, "url": page.url,
                "content": content, "fetched_date": FETCHED_DATE,
            })
        except Exception:
            failed.append(topic)
    except Exception:
        failed.append(topic)
    time.sleep(0.3)

docs = [preprocess_doc(d) for d in raw_docs]

with open("corpus_raw.json", "w", encoding="utf-8") as f:
    json.dump(docs, f, ensure_ascii=False, indent=2)

orig_chars  = sum(len(d["content"]) for d in raw_docs)
clean_chars = sum(len(d["content"]) for d in docs)
print(f"Fetched      : {len(docs)} articles  ({len(failed)} failed)")
print(f"Noise removed: {orig_chars - clean_chars:,} chars ({(orig_chars-clean_chars)/orig_chars*100:.1f}%)")
if failed:
    print(f"Failed topics: {failed}")


## 4. Chunking — Fixed / Recursive / Semantic




In [ ]:
def _ascii_safe_id(text: str) -> str:
    """Return a Pinecone-safe ASCII token from an arbitrary Unicode string."""
    s = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"\s+", "_", s.strip())
    s = re.sub(r"[^A-Za-z0-9_\-]", "", s)
    return s or "untitled"


# ── Strategy A: Fixed ──────────────────────────────────────────────────────────
fixed_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP_FIXED, length_function=len,
)

# ── Strategy B: Recursive ──────────────────────────────────────────────────────
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP_REC, length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],
)


def make_chunks(source_docs, splitter, strategy_name):
    chunks = []
    for doc in source_docs:
        parts    = splitter.split_text(doc["content"])
        safe_src = _ascii_safe_id(doc["title"])
        for i, part in enumerate(parts):
            part = part.strip()
            if len(part) < MIN_CHUNK_CHARS:       # IMPROVED: stricter minimum
                continue
            chunks.append({
                "id"          : f"{strategy_name}_{safe_src}_{i}",
                "text"        : part,
                "source"      : doc["title"],
                "url"         : doc["url"],
                "fetched_date": doc.get("fetched_date", FETCHED_DATE),
                "strategy"    : strategy_name,
                "chunk_no"    : i,
            })
    return chunks


# ── Strategy C: Semantic (FIXED) ───────────────────────────────────────────────
def make_semantic_chunks(source_docs, strategy_name="semantic",
                         max_chars=SEM_MAX_CHARS, min_chars=SEM_MIN_CHARS,
                         similarity_break=SEM_SIM_BREAK,
                         model_name="all-MiniLM-L6-v2"):
    """
    Sentence-level semantic chunking with TWO bug-fixes over the original:

    Fix 1 — Hard-break guard:
        If adding the next sentence would exceed max_chars, flush the current
        buffer REGARDLESS of the similarity score.  This fixes runaway chunks
        produced by stat / honours tables where all adjacent sentences have
        high mutual similarity (same topic = no break ever fires).

    Fix 2 — Per-sentence force-split:
        Individual sentences longer than max_chars are split on whitespace so
        a single sentence can never overflow the buffer by itself.

    MIN_CHUNK_CHARS is also enforced (raised to 80) on every flushed chunk.
    """
    sem_model = SentenceTransformer(model_name)
    chunks    = []

    for doc in tqdm(source_docs, desc="Semantic chunking"):
        text = doc["content"].strip()
        if not text:
            continue

        # Tokenise into sentences; force-split any sentence > max_chars (FIX 2)
        raw_sents = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]
        sentences = []
        for s in raw_sents:
            if len(s) <= max_chars:
                sentences.append(s)
            else:
                words, buf = s.split(), []
                for w in words:
                    if buf and len(" ".join(buf)) + 1 + len(w) > max_chars:
                        sentences.append(" ".join(buf))
                        buf = [w]
                    else:
                        buf.append(w)
                if buf:
                    sentences.append(" ".join(buf))

        if not sentences:
            continue

        embeddings = sem_model.encode(
            sentences, show_progress_bar=False, normalize_embeddings=True
        )

        safe_src = _ascii_safe_id(doc["title"])
        current, current_len, out_idx = [sentences[0]], len(sentences[0]), 0

        def _flush(buf, buf_len, idx):
            chunk_text = " ".join(buf).strip()
            if len(chunk_text) >= MIN_CHUNK_CHARS:
                chunks.append({
                    "id"          : f"{strategy_name}_{safe_src}_{idx}",
                    "text"        : chunk_text,
                    "source"      : doc["title"],
                    "url"         : doc["url"],
                    "fetched_date": doc.get("fetched_date", FETCHED_DATE),
                    "strategy"    : strategy_name,
                    "chunk_no"    : idx,
                })
                return idx + 1
            return idx

        for i in range(1, len(sentences)):
            sim      = float(np.dot(embeddings[i - 1], embeddings[i]))
            sent     = sentences[i]
            sent_len = len(sent)

            would_overflow = (current_len + 1 + sent_len > max_chars)   # FIX 1
            topic_shift    = (current_len >= min_chars and sim < similarity_break)

            if would_overflow or topic_shift:
                out_idx = _flush(current, current_len, out_idx)
                current, current_len = [sent], sent_len
            else:
                current.append(sent)
                current_len += 1 + sent_len

        _flush(current, current_len, out_idx)    # flush final buffer

    return chunks


# ── Run all three ──────────────────────────────────────────────────────────────
fixed_chunks     = make_chunks(docs, fixed_splitter,     "fixed")
recursive_chunks = make_chunks(docs, recursive_splitter, "recursive")
semantic_chunks  = make_semantic_chunks(docs, strategy_name="semantic")

for name, ch in [("fixed", fixed_chunks), ("recursive", recursive_chunks),
                 ("semantic", semantic_chunks)]:
    lengths = [len(c["text"]) for c in ch]
    print(f"{name:10s}: n={len(ch):5d}  avg={sum(lengths)//len(lengths):4d}"
          f"  min={min(lengths):4d}  max={max(lengths):5d} chars")

for fname, ch in [("chunks_fixed.json", fixed_chunks),
                  ("chunks_recursive.json", recursive_chunks),
                  ("chunks_semantic.json", semantic_chunks)]:
    with open(fname, "w", encoding="utf-8") as f:
        json.dump(ch, f, ensure_ascii=False, indent=2)

with open("chunks.json", "w", encoding="utf-8") as f:
    json.dump(recursive_chunks, f, ensure_ascii=False, indent=2)

print("\nSaved: chunks.json  chunks_fixed.json  chunks_recursive.json  chunks_semantic.json")


## 5. Chunk Quality Report

In [ ]:
import statistics as _stats

chunks_by_strategy = {
    "fixed"    : fixed_chunks,
    "recursive": recursive_chunks,
    "semantic" : semantic_chunks,
}

quality_rows = []
for name, ch in chunks_by_strategy.items():
    lengths  = [len(c["text"]) for c in ch]
    texts    = [c["text"] for c in ch]
    quality_rows.append({
        "strategy"       : name,
        "count"          : len(ch),
        "avg_chars"      : round(sum(lengths) / len(lengths)),
        "median_chars"   : int(_stats.median(lengths)),
        "min_chars"      : min(lengths),
        "max_chars"      : max(lengths),
        "short (<100)"   : sum(1 for l in lengths if l < 100),
        "near_max (>900)": sum(1 for l in lengths if l > 900),
        "unique_sources" : len(set(c["source"] for c in ch)),
        "duplicates"     : len(texts) - len(set(texts)),
    })

display(pd.DataFrame(quality_rows))

sem_outliers = [c for c in semantic_chunks if len(c["text"]) > SEM_MAX_CHARS]
if sem_outliers:
    print(f"\nWARNING: {len(sem_outliers)} semantic chunks still exceed "
          f"SEM_MAX_CHARS={SEM_MAX_CHARS}. Check force-split logic.")
    for c in sem_outliers[:3]:
        print(f"  {c['id']}  {len(c['text'])} chars")
else:
    print(f"\nAll semantic chunks are within SEM_MAX_CHARS={SEM_MAX_CHARS}. Fix confirmed.")


## 6. Embed & Upsert to Pinecone

In [ ]:
import torch

if not PINECONE_API_KEY.strip():
    raise ValueError("PINECONE_API_KEY is empty. Set it in the Config cell or Kaggle Secrets.")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading embedding model on {DEVICE}: {EMBEDDING_MODEL}")
_upsert_model = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)

sample_vec    = _upsert_model.encode(["probe"], show_progress_bar=False)
EMBEDDING_DIM = int(len(sample_vec[0]))
print(f"Embedding dimension: {EMBEDDING_DIM}")

# Validate IDs
for name, ch in chunks_by_strategy.items():
    bad = [c["id"] for c in ch if not str(c.get("id","")).isascii()]
    if bad:
        raise ValueError(f"{name}: non-ASCII chunk ID: {bad[0]}")

# Create / validate Pinecone index
pc = Pinecone(api_key=PINECONE_API_KEY)
existing = [idx.name for idx in pc.list_indexes()]

if INDEX_NAME not in existing:
    pc.create_index(
        name=INDEX_NAME, dimension=EMBEDDING_DIM, metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print(f"Index '{INDEX_NAME}' created (dim={EMBEDDING_DIM}).")
else:
    info = pc.describe_index(INDEX_NAME)
    dim  = info.dimension if hasattr(info, "dimension") else info["dimension"]
    if int(dim) != EMBEDDING_DIM:
        raise ValueError(
            f"Index dim={dim} mismatches model dim={EMBEDDING_DIM}. "
            "Delete the index or change INDEX_NAME."
        )
    print(f"Index '{INDEX_NAME}' exists (dim={dim}).")

pc_index = pc.Index(INDEX_NAME)


def upsert_chunks(chunks, namespace):
    print(f"\nUpserting namespace='{namespace}' ({len(chunks)} chunks)…")
    for i in tqdm(range(0, len(chunks), BATCH_SIZE), desc=f"  {namespace}"):
        batch  = chunks[i : i + BATCH_SIZE]
        texts  = [c["text"] for c in batch]
        embs   = _upsert_model.encode(
            texts, show_progress_bar=False, normalize_embeddings=True
        ).tolist()
        vectors = [
            (c["id"], emb, {
                "text"        : c["text"],
                "source"      : c["source"],
                "url"         : c["url"],
                "fetched_date": c.get("fetched_date", ""),
                "strategy"    : c.get("strategy", namespace),
            })
            for c, emb in zip(batch, embs)
        ]
        pc_index.upsert(vectors=vectors, namespace=namespace)


for strategy_name, strategy_chunks in chunks_by_strategy.items():
    upsert_chunks(strategy_chunks, namespace=strategy_name)

print("\nAll namespaces upserted.")
print(pc_index.describe_index_stats())


## 7. Pinecone Sanity Check

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
embed_model = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)


def _query_instruction(q: str) -> str:
    """BGE retrieval instruction prefix."""
    return f"Represent this sentence for searching relevant passages: {q}"


test_query = "Who won the FIFA World Cup in 2022?"
test_vec   = embed_model.encode(
    _query_instruction(test_query), normalize_embeddings=True
).tolist()

for ns in ["fixed", "recursive", "semantic"]:
    results = pc_index.query(
        vector=test_vec, top_k=3, include_metadata=True, namespace=ns
    )
    print(f"\n=== namespace={ns} ===")
    for i, m in enumerate(results.get("matches", []), 1):
        meta = m.get("metadata", {})
        print(f"  [{i}] score={m['score']:.4f}  source={meta.get('source','?')}  "
              f"fetched={meta.get('fetched_date','?')}")
        print(f"       {meta.get('text','')[:120]}...")


## 8. BM25 Index Build

In [ ]:
def _load_chunks_if_needed(name):
    """Load chunk JSON from disk; tries CWD then Kaggle input paths."""
    path = f"chunks_{name}.json"
    if not Path(path).exists() and Path("/kaggle/input").exists():
        for p in Path("/kaggle/input").rglob(f"chunks_{name}.json"):
            path = str(p)
            break
    with open(path, encoding="utf-8") as f:
        return json.load(f)


# Re-load from disk if not already in memory (e.g. if skipping Section 4)
chunks_by_strategy = {
    s: globals().get(f"{s}_chunks") or _load_chunks_if_needed(s)
    for s in ["fixed", "recursive", "semantic"]
}

chunk_lookup = {
    s: {c["id"]: c for c in ch}
    for s, ch in chunks_by_strategy.items()
}

def bm25_tokenize(text: str):
    return re.findall(r"[A-Za-z0-9]+", text.lower())

bm25_indexes = {}
for s, ch in chunks_by_strategy.items():
    corpus = [bm25_tokenize(c["text"]) for c in ch]
    bm25_indexes[s] = BM25Okapi(corpus)
    print(f"BM25 index built for '{s}' ({len(ch)} chunks).")


## 9. Retrieval Helpers

### Pipeline: Hybrid (BM25 + Dense → RRF) → CrossEncoder → MMR




In [ ]:
import time as _time

# ── BM25 retrieval ─────────────────────────────────────────────────────────────
def bm25_retrieve(query: str, strategy: str, top_k: int = TOP_K):
    ch     = chunks_by_strategy[strategy]
    bm25   = bm25_indexes[strategy]
    tokens = bm25_tokenize(query)
    scores = bm25.get_scores(tokens)
    idx    = np.argsort(scores)[::-1][:top_k]
    return [{
        "id"          : ch[i]["id"],
        "score"       : float(scores[i]),
        "text"        : ch[i].get("text", ""),
        "source"      : ch[i].get("source", ""),
        "url"         : ch[i].get("url", ""),
        "fetched_date": ch[i].get("fetched_date", ""),
        "strategy"    : strategy,
    } for i in idx]


# ── Dense (Pinecone) retrieval ────────────────────────────────────────────────
def dense_retrieve(query: str, namespace: str, strategy: str,
                   top_k: int = TOP_K):
    q_vec    = embed_model.encode(
        _query_instruction(query), normalize_embeddings=True
    ).tolist()
    response = pc_index.query(
        vector=q_vec, top_k=top_k, namespace=namespace, include_metadata=True,
    )
    local   = chunk_lookup[strategy]
    results = []
    for m in response.get("matches", []):
        mid  = m.get("id")
        meta = m.get("metadata", {}) or {}
        fb   = local.get(mid, {})
        results.append({
            "id"          : mid,
            "score"       : float(m.get("score", 0.0)),
            "text"        : meta.get("text",         fb.get("text",  "")),
            "source"      : meta.get("source",       fb.get("source","")),
            "url"         : meta.get("url",          fb.get("url",   "")),
            "fetched_date": meta.get("fetched_date", fb.get("fetched_date","")),
            "strategy"    : strategy,
        })
    return results


# ── Reciprocal Rank Fusion ─────────────────────────────────────────────────────
def reciprocal_rank_fusion(result_lists, rrf_k: int = RRF_K, top_k: int = TOP_K):
    fused, payload = defaultdict(float), {}
    for results in result_lists:
        for rank, item in enumerate(results, start=1):
            did = item["id"]
            fused[did] += 1.0 / (rrf_k + rank)
            payload.setdefault(did, item)
    ranked = sorted(fused.items(), key=lambda x: x[1], reverse=True)[:top_k]
    return [{**payload[did], "score": score} for did, score in ranked]


# ── Hybrid retrieval ───────────────────────────────────────────────────────────
def hybrid_retrieve(query: str, strategy: str, top_k: int = TOP_K):
    sparse = bm25_retrieve(query, strategy=strategy, top_k=DENSE_TOP_K)
    dense  = dense_retrieve(query, namespace=strategy, strategy=strategy,
                            top_k=DENSE_TOP_K)
    return reciprocal_rank_fusion([sparse, dense], rrf_k=RRF_K, top_k=top_k)


# ── Cross-encoder reranking ────────────────────────────────────────────────────
_ce_model = None

def get_cross_encoder():
    global _ce_model
    if _ce_model is None:
        print(f"Loading cross-encoder: {CROSS_ENCODER_MODEL}")
        _ce_model = CrossEncoder(CROSS_ENCODER_MODEL)
    return _ce_model


def cross_encoder_rerank(query: str, candidates, top_k: int = TOP_K):
    if not candidates:
        return []
    ce     = get_cross_encoder()
    pairs  = [[query, c.get("text", "")] for c in candidates]
    scores = ce.predict(pairs)
    ranked = sorted(
        [{**c, "rerank_score": float(s)} for c, s in zip(candidates, scores)],
        key=lambda x: x["rerank_score"], reverse=True,
    )
    return ranked[:top_k]


# ── MMR deduplication (NEW) ────────────────────────────────────────────────────
def mmr_deduplicate(query: str, candidates, top_k: int = QA_CONTEXT_TOP_K,
                    lambda_mmr: float = MMR_LAMBDA):
    """
    Maximal Marginal Relevance selection.

    Iteratively selects the candidate that maximises:
        lambda_mmr * sim(doc, query) - (1 - lambda_mmr) * max_sim(doc, selected)

    This trades a small amount of relevance for greater source diversity,
    preventing context repetition when multiple top chunks come from the same
    Wikipedia article.

    lambda_mmr = 1.0  →  pure relevance (MMR disabled, same as slicing [:top_k])
    lambda_mmr = 0.0  →  pure diversity (maximise coverage of different sources)
    Default MMR_LAMBDA = 0.6 gives a balanced result.
    """
    if len(candidates) <= top_k:
        return candidates

    texts = [c.get("text", "") for c in candidates]
    vecs  = embed_model.encode(texts, normalize_embeddings=True)
    q_vec = embed_model.encode(
        _query_instruction(query), normalize_embeddings=True
    )

    rel_scores   = vecs @ q_vec
    selected_idx = []
    remaining    = list(range(len(candidates)))

    for _ in range(top_k):
        if not remaining:
            break
        if not selected_idx:
            best = max(remaining, key=lambda i: rel_scores[i])
        else:
            sel_vecs = vecs[selected_idx]
            best, best_score = None, -1e9
            for i in remaining:
                max_sim  = float(np.max(sel_vecs @ vecs[i]))
                mmr_sc   = lambda_mmr * rel_scores[i] - (1 - lambda_mmr) * max_sim
                if mmr_sc > best_score:
                    best, best_score = i, mmr_sc
        selected_idx.append(best)
        remaining.remove(best)

    return [candidates[i] for i in selected_idx]


# ── Full pipeline convenience function ────────────────────────────────────────
def retrieve_and_rerank(query: str, strategy: str,
                        n_candidates: int = RERANK_CANDIDATES_TOP_K,
                        final_k: int = QA_CONTEXT_TOP_K):
    """
    Full retrieval pipeline:
      1. Hybrid (BM25 + Dense → RRF) — n_candidates results
      2. Cross-encoder reranking
      3. MMR deduplication → final_k context chunks
    """
    candidates = hybrid_retrieve(query, strategy=strategy, top_k=n_candidates)
    reranked   = cross_encoder_rerank(query, candidates, top_k=n_candidates)
    return mmr_deduplicate(query, reranked, top_k=final_k)


# ── Quick sanity check ────────────────────────────────────────────────────────
_sq = "Who won the 2022 FIFA World Cup final?"
print("BM25 recursive (top-3):")
for r in bm25_retrieve(_sq, "recursive", top_k=3):
    print(f"  {r['id']}  score={r['score']:.3f}")


## 10. Evaluation Set — Curated Gold + Auto-generated

### Why a curated gold set?
The original auto-generation builds questions *from chunk entity names* (e.g. `"What competition is UEFA Champions League?"`). Because the entity name appears verbatim in the chunk ID, BM25 retrieves the correct chunk nearly trivially, **inflating Recall and MRR metrics**.

25 hand-written questions with specific, non-trivial answers are merged with the auto-generated set. The gold questions test cross-entity comparison, multi-hop reasoning, and depth of knowledge — things that genuinely exercise the retrieval stack.


In [ ]:
# ── 25 curated gold questions ──────────────────────────────────────────────────
GOLD_QUESTIONS = [
    # Factual / entity-level
    dict(question="Which country has won the most FIFA World Cup titles in history?",
         question_type="competition", expect_abstain=False),
    dict(question="How many UEFA Champions League titles has Real Madrid won?",
         question_type="club", expect_abstain=False),
    dict(question="Who scored the winning goal in the 2022 FIFA World Cup final penalty shootout?",
         question_type="person", expect_abstain=False),
    dict(question="Which player has won the Ballon d'Or award the most times?",
         question_type="person", expect_abstain=False),
    dict(question="What year did Argentina win the Copa America for the very first time?",
         question_type="country-team", expect_abstain=False),
    dict(question="Which English club has won the Premier League the most times?",
         question_type="club", expect_abstain=False),
    dict(question="Who is the all-time top scorer in UEFA Champions League history?",
         question_type="person", expect_abstain=False),
    dict(question="What was the score in the 1970 FIFA World Cup final between Brazil and Italy?",
         question_type="competition", expect_abstain=False),
    dict(question="Which club did Zinedine Zidane win the Champions League with as a player?",
         question_type="person", expect_abstain=False),
    dict(question="How many goals did Ronaldo (the Brazilian) score across all World Cup tournaments?",
         question_type="person", expect_abstain=False),
    # Multi-hop / reasoning
    dict(question="Which rule change did VAR introduce to how offside decisions are made?",
         question_type="general", expect_abstain=False),
    dict(question="Which Bundesliga club has won the most consecutive German league titles?",
         question_type="club", expect_abstain=False),
    dict(question="What distinguishes the Copa Libertadores from the Copa America?",
         question_type="competition", expect_abstain=False),
    dict(question="What is the prize money difference between the Premier League and La Liga?",
         question_type="competition", expect_abstain=True),   # unlikely in corpus
    dict(question="How does Reciprocal Rank Fusion combine sparse and dense retrieval scores?",
         question_type="general", expect_abstain=True),        # outside football corpus
    # Cross-entity comparison
    dict(question="Which club has more Champions League titles — AC Milan or Liverpool?",
         question_type="club", expect_abstain=False),
    dict(question="Did Messi or Ronaldo score more goals in a single calendar year?",
         question_type="person", expect_abstain=False),
    dict(question="Which national team knocked Germany out of the 2018 FIFA World Cup group stage?",
         question_type="country-team", expect_abstain=False),
    dict(question="How many World Cup goals did Pele score compared to Diego Maradona?",
         question_type="person", expect_abstain=False),
    dict(question="Which club signed Erling Haaland before he moved to Manchester City?",
         question_type="person", expect_abstain=False),
    # Harder in-domain (tests depth)
    dict(question="When was the offside rule last significantly amended in association football?",
         question_type="general", expect_abstain=False),
    dict(question="What formation did Spain predominantly use when winning UEFA Euro 2008?",
         question_type="country-team", expect_abstain=False),
    dict(question="How many Serie A titles has Juventus won in total?",
         question_type="club", expect_abstain=False),
    dict(question="What is the format of the UEFA Conference League knockout rounds?",
         question_type="competition", expect_abstain=False),
    dict(question="Which stadium hosted the first ever FIFA World Cup match in 1930?",
         question_type="competition", expect_abstain=False),
]

gold_items = [
    {**q, "relevant_chunk_ids": [], "source": "gold"}
    for q in GOLD_QUESTIONS
]
print(f"Gold questions ready: {len(gold_items)}")


In [ ]:
# ── Auto-generated questions ───────────────────────────────────────────────────
def _parse_slug(chunk_id: str, prefix: str):
    m = re.match(rf"^{prefix}_(.*)_\d+$", chunk_id)
    return m.group(1) if m else None

def _pretty(slug: str) -> str:
    return slug.replace("_", " ").strip()

def _qtype(name: str, text: str) -> str:
    n, t = name.lower(), text.lower()
    if "national football team" in n:                                    return "country-team"
    if any(k in n for k in ["cup","league","bundesliga","eredivisie","libertadores","champions"]):
        return "competition"
    if any(k in t for k in ["footballer","striker","midfielder","goalkeeper","player","manager"]):
        return "person"
    if any(k in n for k in ["f.c","fc","milan","arsenal","chelsea","dortmund","barcelona","bayern","ajax"]):
        return "club"
    return "general"

def _variants(name: str, qtype: str) -> list:
    if qtype == "country-team":
        return [f"In international football, what country is {name} associated with?",
                f"{name} represents which nation?"]
    if qtype == "competition":
        return [f"What type of tournament is {name}?",
                f"How is {name} categorized in football?"]
    if qtype == "club":
        return [f"What is {name} known for in club football?",
                f"In football, how would you describe {name}?"]
    if qtype == "person":
        return [f"What is {name} known for in football?",
                f"Give a brief football profile of {name}."]
    return [f"Give a brief overview of {name} in football."]


id_by_entity   = defaultdict(dict)
text_by_entity = {}

for strat in ["fixed", "recursive", "semantic"]:
    for item in chunks_by_strategy[strat]:
        slug = _parse_slug(item["id"], strat)
        if not slug:
            continue
        id_by_entity[slug].setdefault(strat, item["id"])
        text_by_entity.setdefault(slug, item.get("text",""))

common_entities = sorted(
    e for e, m in id_by_entity.items()
    if {"fixed","recursive","semantic"}.issubset(m.keys())
)

auto_target  = TARGET_EVAL_QUESTIONS - UNANSWERABLE_QUESTIONS - len(gold_items)
auto_items   = []
used_norms   = set()

for slug in common_entities:
    if len(auto_items) >= auto_target:
        break
    name    = _pretty(slug)
    qtype   = _qtype(name, text_by_entity.get(slug,""))
    rel_ids = [id_by_entity[slug][s] for s in ["fixed","recursive","semantic"]]
    for q in _variants(name, qtype):
        if q.strip().lower() not in used_norms:
            used_norms.add(q.strip().lower())
            auto_items.append({
                "question"          : q,
                "relevant_chunk_ids": rel_ids,
                "question_type"     : qtype,
                "expect_abstain"    : False,
                "source"            : "auto",
            })
            if len(auto_items) >= auto_target:
                break

print(f"Auto-generated answerable questions: {len(auto_items)}")

# ── Unanswerable questions ─────────────────────────────────────────────────────
UNANSWERABLE = [
    "Who won the UEFA Euro 2024 Golden Boot?",
    "What was Lionel Messi's salary at Inter Miami in 2025?",
    "Who scored the fastest goal in the FIFA World Cup 2030?",
    "Which team won the 2026 AFC Champions League final?",
    "What is the transfer fee for Kylian Mbappe in 2026?",
    "Who is the top scorer of the Saudi Pro League 2025-26 season?",
    "Which club signed Jude Bellingham in summer 2027?",
    "What was the exact xG of the 2024 Champions League final?",
    "Who won the Ballon d'Or in 2026?",
    "What was the attendance of the 2025 Club World Cup final?",
    "Which city hosted the 2028 FIFA World Cup opening match?",
    "Who was the referee for the 2029 Copa Libertadores final?",
    "What was Pep Guardiola's win rate across all competitions in 2025?",
    "Which footballer announced retirement in January 2027?",
    "What VAR decision was controversial in the 2026 UCL final?",
]
unans_items = [{
    "question": q, "relevant_chunk_ids": [], "question_type": "unanswerable",
    "expect_abstain": True, "source": "manual",
} for q in UNANSWERABLE]

# ── Combine & shuffle ──────────────────────────────────────────────────────────
evaluation_questions = gold_items + auto_items + unans_items
random.seed(42)
random.shuffle(evaluation_questions)

eval_df = pd.DataFrame(evaluation_questions)
print(f"\nTotal evaluation questions : {len(evaluation_questions)}")
print(f"  Gold (curated)           : {sum(1 for x in evaluation_questions if x.get('source')=='gold')}")
print(f"  Auto-generated           : {sum(1 for x in evaluation_questions if x.get('source')=='auto')}")
print(f"  Unanswerable             : {sum(1 for x in evaluation_questions if x['expect_abstain'])}")
print("\nQuestion type distribution:")
display(eval_df["question_type"].value_counts().rename_axis("type").to_frame("count"))


## 11. Retrieval Experiments — All 6 Configurations

In [ ]:
def compute_recall_at_k(pred_ids, rel_ids, k):
    return 1.0 if set(pred_ids[:k]).intersection(rel_ids) else 0.0

def compute_mrr_at_k(pred_ids, rel_ids, k=10):
    for rank, did in enumerate(pred_ids[:k], 1):
        if did in set(rel_ids):
            return 1.0 / rank
    return 0.0


def evaluate_retriever(eval_data, retriever_fn, top_k=10, label=""):
    rows = []
    for item in tqdm(eval_data, desc=f"Eval {label}"):
        q              = item["question"]
        rel            = item.get("relevant_chunk_ids", [])
        expect_abstain = bool(item.get("expect_abstain") or not rel)
        t0             = _time.perf_counter()
        results        = retriever_fn(q, top_k)
        elapsed        = _time.perf_counter() - t0
        pred_ids       = [r["id"] for r in results]
        rows.append({
            "question"          : q,
            "question_type"     : item.get("question_type","unknown"),
            "source"            : item.get("source","auto"),
            "expect_abstain"    : expect_abstain,
            "recall@5"  : compute_recall_at_k(pred_ids, rel, 5)  if not expect_abstain else np.nan,
            "recall@10" : compute_recall_at_k(pred_ids, rel, 10) if not expect_abstain else np.nan,
            "mrr@10"    : compute_mrr_at_k(pred_ids, rel, 10)    if not expect_abstain else np.nan,
            "retrieval_time_sec": elapsed,
        })
    df  = pd.DataFrame(rows)
    ans = df[~df["expect_abstain"]]
    return {
        "Recall@5"               : ans["recall@5"].mean()  if not ans.empty else np.nan,
        "Recall@10"              : ans["recall@10"].mean() if not ans.empty else np.nan,
        "MRR@10"                 : ans["mrr@10"].mean()    if not ans.empty else np.nan,
        "Answerable Count"       : len(ans),
        "Avg Retrieval Time (s)" : df["retrieval_time_sec"].mean(),
        "P95 Retrieval Time (s)" : df["retrieval_time_sec"].quantile(0.95),
        "details"                : df,
    }


dense_ready = bool(PINECONE_API_KEY.strip() or os.environ.get("PINECONE_API_KEY",""))

experiments = {
    "bm25_fixed"    : lambda q, k: bm25_retrieve(q, "fixed",     top_k=k),
    "bm25_recursive": lambda q, k: bm25_retrieve(q, "recursive", top_k=k),
    "bm25_semantic" : lambda q, k: bm25_retrieve(q, "semantic",  top_k=k),
}
if dense_ready:
    experiments.update({
        "hybrid_fixed"    : lambda q, k: hybrid_retrieve(q, "fixed",     top_k=k),
        "hybrid_recursive": lambda q, k: hybrid_retrieve(q, "recursive", top_k=k),
        "hybrid_semantic" : lambda q, k: hybrid_retrieve(q, "semantic",  top_k=k),
    })
else:
    print("PINECONE_API_KEY not set — BM25-only experiments.")

all_results             = []
retrieval_detail_by_exp = {}

for name, rfn in experiments.items():
    m = evaluate_retriever(evaluation_questions, rfn, top_k=10, label=name)
    all_results.append({
        "experiment"       : name,
        "Recall@5"         : m["Recall@5"],
        "Recall@10"        : m["Recall@10"],
        "MRR@10"           : m["MRR@10"],
        "answerable"       : m["Answerable Count"],
        "avg_retrieval_sec": m["Avg Retrieval Time (s)"],
        "p95_retrieval_sec": m["P95 Retrieval Time (s)"],
    })
    retrieval_detail_by_exp[name] = m["details"]

results_df = (
    pd.DataFrame(all_results)
    .sort_values(["MRR@10","avg_retrieval_sec"], ascending=[False, True])
    .reset_index(drop=True)
)
display(results_df)

# ── Per-category MRR@10 breakdown (NEW) ───────────────────────────────────────
print("\n=== Per-category MRR@10 ===")
for name, detail_df in retrieval_detail_by_exp.items():
    ans = detail_df[~detail_df["expect_abstain"]]
    print(f"\n{name}")
    print(ans.groupby("question_type")["mrr@10"].mean().round(3).to_string())

# ── Gold vs auto breakdown ─────────────────────────────────────────────────────
print("\n=== Gold vs Auto MRR@10 (best experiment) ===")
best_exp_name = results_df.iloc[0]["experiment"]
bdf = retrieval_detail_by_exp[best_exp_name]
for src_type in ["gold","auto"]:
    subset = bdf[(bdf["source"]==src_type) & (~bdf["expect_abstain"])]
    if not subset.empty:
        print(f"  {src_type}: MRR@10={subset['mrr@10'].mean():.3f}  (n={len(subset)})")


## 12. Winner Selection

In [ ]:
sparse_df = results_df[results_df["experiment"].str.startswith("bm25_")]
hybrid_df = results_df[results_df["experiment"].str.startswith("hybrid_")]

best_sparse = sparse_df.iloc[0]["experiment"] if not sparse_df.empty else None
best_hybrid = hybrid_df.iloc[0]["experiment"] if not hybrid_df.empty else None

print(f"Best sparse retriever: {best_sparse}")
print(f"Best hybrid retriever: {best_hybrid}")

_strategy_map = {
    "bm25_fixed": "fixed", "bm25_recursive": "recursive", "bm25_semantic": "semantic",
    "hybrid_fixed": "fixed", "hybrid_recursive": "recursive", "hybrid_semantic": "semantic",
}


## 13. QA Generation (Groq)

In [ ]:
from openai import OpenAI, RateLimitError as _RLE

def _init_groq():
    key = os.environ.get("GROQ_API_KEY","")
    if not key:
        try:
            from kaggle_secrets import UserSecretsClient
            key = UserSecretsClient().get_secret("GROQ_API_KEY") or ""
            if key:
                os.environ["GROQ_API_KEY"] = key
        except Exception:
            pass
    if not key:
        raise EnvironmentError("GROQ_API_KEY not found.")
    return OpenAI(api_key=key, base_url="https://api.groq.com/openai/v1")

groq_client = _init_groq()


_RECENCY_KEYWORDS = {"current","now","today","latest","recent","2025","2026","2027","2028"}

def _may_be_stale(question: str) -> bool:
    return any(kw in question.lower() for kw in _RECENCY_KEYWORDS)


def build_context(hits, max_chars=2500) -> str:
    parts, total = [], 0
    for r in hits:
        txt = r.get("text","")
        if not txt or total + len(txt) > max_chars:
            break
        parts.append(txt)
        total += len(txt)
    return "\n\n".join(parts)


def make_qa_prompt(question: str, context: str) -> str:
    return (
        "Answer the question using ONLY the context provided below.\n"
        "Write a complete, fluent sentence.\n"
        "If the context does not contain enough information, "
        "respond with exactly: I don't know.\n\n"
        f"Question: {question}\n\n"
        f"Context:\n{context}\n\n"
        "Answer:"
    )

def is_abstention(answer: str) -> bool:
    pats = ["i don't know","i do not know","insufficient context",
            "not enough context","cannot determine","can't determine","unknown"]
    return any(p in (answer or "").strip().lower() for p in pats)


def generate_answer(question: str, context: str, max_tokens=200) -> str:
    prompt = make_qa_prompt(question, context)
    for attempt in range(6):
        try:
            resp = groq_client.chat.completions.create(
                model=QA_GEN_MODEL, temperature=0, max_tokens=max_tokens,
                messages=[{"role":"user","content":prompt}],
            )
            return resp.choices[0].message.content.strip()
        except _RLE:
            if attempt == 5: raise
            wait = 2.0 * (2 ** attempt)
            print(f"  Rate limit — waiting {wait:.0f}s…")
            _time.sleep(wait)


def answer_with_pipeline(question: str, strategy: str, use_hybrid: bool = True):
    t0 = _time.perf_counter()
    hits   = retrieve_and_rerank(question, strategy) if use_hybrid else              dense_retrieve(question, namespace=strategy, strategy=strategy,
                            top_k=QA_CONTEXT_TOP_K)
    t_ret  = _time.perf_counter() - t0
    context = build_context(hits)
    t1      = _time.perf_counter()
    answer  = generate_answer(question, context)
    t_inf   = _time.perf_counter() - t1
    return {
        "answer": answer, "context": context, "hits": hits,
        "stale_warning": _may_be_stale(question),
        "retrieval_time_sec": t_ret, "inference_time_sec": t_inf,
        "overall_response_time": t_ret + t_inf,
    }


# ── QA evaluation ──────────────────────────────────────────────────────────────
winner_map = {}
for w in [best_sparse, best_hybrid]:
    if w and w in experiments:
        winner_map[w] = _strategy_map.get(w, "recursive")

qa_eval_qs = evaluation_questions[:min(QA_EVAL_SIZE, len(evaluation_questions))]
qa_rows, qa_arts = [], []

for item in tqdm(qa_eval_qs, desc="QA eval"):
    q              = item["question"]
    expect_abstain = bool(item.get("expect_abstain") or not item.get("relevant_chunk_ids",[]))
    row = {"question": q, "expected_abstain": expect_abstain,
           "question_type": item.get("question_type","unknown")}

    for w_name, strat in winner_map.items():
        res     = answer_with_pipeline(q, strategy=strat, use_hybrid=w_name.startswith("hybrid_"))
        pred_abs = is_abstention(res["answer"])
        row[f"answer_{w_name}"]          = res["answer"]
        row[f"stale_warning_{w_name}"]   = res["stale_warning"]
        row[f"ret_time_{w_name}"]        = res["retrieval_time_sec"]
        row[f"inf_time_{w_name}"]        = res["inference_time_sec"]
        row[f"total_time_{w_name}"]      = res["overall_response_time"]
        row[f"abstain_correct_{w_name}"] = (pred_abs == expect_abstain)
        qa_arts.append({
            "question": q, "question_type": item.get("question_type"),
            "winner": w_name, "strategy": strat,
            "expected_abstain": expect_abstain, "predicted_abstain": pred_abs,
            "abstain_correct": (pred_abs == expect_abstain),
            "answer": res["answer"], "context": res["context"],
            "stale_warning": res["stale_warning"],
            "retrieval_time_sec": res["retrieval_time_sec"],
            "inference_time_sec": res["inference_time_sec"],
            "overall_response_time_sec": res["overall_response_time"],
        })
        _time.sleep(0.2)
    qa_rows.append(row)

qa_df     = pd.DataFrame(qa_rows)
qa_arts_df = pd.DataFrame(qa_arts)

summary = (
    qa_arts_df.groupby("winner", as_index=False)
    .agg(
        questions        =("question","count"),
        abstain_accuracy =("abstain_correct","mean"),
        avg_retrieval_sec=("retrieval_time_sec","mean"),
        avg_inference_sec=("inference_time_sec","mean"),
        avg_total_sec    =("overall_response_time_sec","mean"),
        p95_total_sec    =("overall_response_time_sec", lambda s: s.quantile(0.95)),
    )
    .sort_values("avg_total_sec").reset_index(drop=True)
)
display(summary)


## 14. LLM-as-Judge — Faithfulness & Response Relevancy

Evaluates the best hybrid retriever. Shows per-category scores and 3 worked example queries.


In [ ]:
def llm_json(client, sys_p, usr_p, model=JUDGE_MODEL_NAME, retries=6, base_delay=2.0):
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=model, temperature=0, response_format={"type":"json_object"},
                messages=[{"role":"system","content":sys_p},
                          {"role":"user",  "content":usr_p}],
            )
            return json.loads(resp.choices[0].message.content)
        except _RLE:
            if attempt == retries - 1: raise
            wait = base_delay * (2 ** attempt)
            print(f"  Rate limit — waiting {wait:.0f}s…")
            _time.sleep(wait)
        except Exception:
            raise


def extract_claims(client, answer: str) -> list:
    data = llm_json(client,
        "You extract factual atomic claims from answers.",
        "Extract atomic factual claims from the answer. "
        "Return JSON with key 'claims' as a list of short strings. "
        "Return an empty list if the answer contains no factual content.\n\n"
        f"Answer:\n{answer}")
    claims = data.get("claims", [])
    return [str(c).strip() for c in (claims if isinstance(claims, list) else []) if str(c).strip()]


def verify_claim(client, question: str, claim: str, context: str) -> dict:
    data = llm_json(client,
        "You verify whether a claim is supported by provided context only.",
        "Verify if the claim is supported by the context only. "
        "Return JSON with keys: label ('supported' or 'unsupported'), "
        "rationale (1-2 lines), evidence (short phrase or empty).\n\n"
        f"Question: {question}\n\nClaim: {claim}\n\nContext:\n{context}")
    label = str(data.get("label","unsupported")).strip().lower()
    if label not in {"supported","unsupported"}:
        label = "unsupported"
    return {"claim": claim, "label": label,
            "rationale": str(data.get("rationale","")).strip(),
            "evidence" : str(data.get("evidence","")).strip()}


def generate_alt_questions(client, answer: str) -> list:
    data = llm_json(client,
        "You generate alternative user queries that the given answer could respond to.",
        "Generate exactly 3 different questions this answer could respond to. "
        "Return JSON with key 'questions' as a list of exactly 3 strings.\n\n"
        f"Answer:\n{answer}")
    qs = data.get("questions", [])
    return [str(q).strip() for q in (qs if isinstance(qs, list) else []) if str(q).strip()][:3]


def cosine_sim_str(a: str, b: str) -> float:
    vecs = embed_model.encode([a, b], normalize_embeddings=True)
    return float(np.dot(vecs[0], vecs[1]))


# ── Select judge target ────────────────────────────────────────────────────────
judge_target = best_hybrid or best_sparse
if judge_target is None:
    raise ValueError("No winner retriever available for judge evaluation.")

judge_strat = _strategy_map.get(judge_target, "recursive")
judge_qs    = [
    item["question"] for item in evaluation_questions
    if not item.get("expect_abstain")
][:min(JUDGE_EVAL_SIZE, 9999)]

judge_input = qa_arts_df[
    (qa_arts_df["winner"]   == judge_target) &
    (qa_arts_df["question"].isin(judge_qs))
].copy()

judge_rows, example_rows = [], []

for _, row in tqdm(judge_input.iterrows(), total=len(judge_input), desc="LLM-as-Judge"):
    question = row["question"]
    answer   = row["answer"]
    context  = row["context"]
    qtype    = row.get("question_type", "unknown")

    claims        = extract_claims(groq_client, answer)
    verifications = []
    for claim in claims:
        verifications.append(verify_claim(groq_client, question, claim, context))
        _time.sleep(0.15)

    supported    = sum(1 for v in verifications if v["label"] == "supported")
    n_claims     = len(verifications)
    faithfulness = supported / n_claims if n_claims else 0.0

    alt_qs    = generate_alt_questions(groq_client, answer)
    sims      = [cosine_sim_str(question, aq) for aq in alt_qs]
    relevancy = float(np.mean(sims)) if sims else 0.0

    judge_rows.append({
        "question"           : question,
        "question_type"      : qtype,
        "winner"             : judge_target,
        "n_claims"           : n_claims,
        "supported_claims"   : supported,
        "faithfulness_score" : faithfulness,
        "alt_questions"      : json.dumps(alt_qs,  ensure_ascii=False),
        "alt_similarities"   : json.dumps(sims,    ensure_ascii=False),
        "avg_relevancy_score": relevancy,
    })

    if len(example_rows) < 3:
        example_rows.append({
            "question"            : question,
            "answer"              : answer,
            "claims"              : json.dumps(claims,         ensure_ascii=False),
            "verification_results": json.dumps(verifications,  ensure_ascii=False),
            "alt_questions"       : json.dumps(alt_qs,         ensure_ascii=False),
            "faithfulness_score"  : round(faithfulness, 4),
            "avg_relevancy_score" : round(relevancy, 4),
        })
    _time.sleep(0.2)

judge_df      = pd.DataFrame(judge_rows)
judge_examples = pd.DataFrame(example_rows)

if not judge_df.empty:
    print(f"Queries judged : {len(judge_df)}")
    print(f"Avg Faithfulness : {judge_df['faithfulness_score'].mean():.4f}")
    print(f"Avg Relevancy    : {judge_df['avg_relevancy_score'].mean():.4f}")

    print("\n=== Per-category Judge Scores (NEW) ===")
    display(
        judge_df.groupby("question_type")[["faithfulness_score","avg_relevancy_score"]]
        .mean().round(4)
    )

print("\n=== 3 Example Judge Queries ===")
display(judge_examples[["question","faithfulness_score","avg_relevancy_score",
                          "claims","alt_questions"]])


## 15. Ablation Study — Chunking × Retrieval

Compares **3 chunking strategies × 2 retrieval variants** on Faithfulness, Relevancy, and latency metrics.




In [ ]:
if not dense_ready:
    raise EnvironmentError("Ablation requires Pinecone. Set PINECONE_API_KEY and rerun.")

ABLATION_VARIATIONS = [
    ("fixed",     "semantic_only"),
    ("fixed",     "hybrid_plus_rerank"),
    ("recursive", "semantic_only"),
    ("recursive", "hybrid_plus_rerank"),
    ("semantic",  "semantic_only"),
    ("semantic",  "hybrid_plus_rerank"),
]

abl_qs = [
    item["question"] for item in evaluation_questions
    if not item.get("expect_abstain")
][:ABLATION_EVAL_SIZE]

ablation_rows = []

for question in tqdm(abl_qs, desc="Ablation"):
    for chunking, variant in ABLATION_VARIATIONS:
        t0 = _time.perf_counter()

        t_r0 = _time.perf_counter()
        if variant == "semantic_only":
            hits = dense_retrieve(question, namespace=chunking, strategy=chunking,
                                  top_k=QA_CONTEXT_TOP_K)
        else:
            hits = retrieve_and_rerank(question, strategy=chunking)   # includes MMR
        t_ret = _time.perf_counter() - t_r0

        context = build_context(hits)
        t_i0    = _time.perf_counter()
        answer  = generate_answer(question, context)
        t_inf   = _time.perf_counter() - t_i0
        overall = _time.perf_counter() - t0
        _time.sleep(0.2)

        claims = extract_claims(groq_client, answer)
        vers   = []
        for claim in claims:
            vers.append(verify_claim(groq_client, question, claim, context))
            _time.sleep(0.15)

        supported    = sum(1 for v in vers if v["label"]=="supported")
        n_claims     = len(vers)
        faithfulness = supported / n_claims if n_claims else 0.0

        alt_qs_ab = generate_alt_questions(groq_client, answer)
        sims_ab   = [cosine_sim_str(question, aq) for aq in alt_qs_ab]
        relevancy = float(np.mean(sims_ab)) if sims_ab else 0.0

        ablation_rows.append({
            "question"           : question,
            "chunking"           : chunking,
            "retrieval_variant"  : variant,
            "answer"             : answer,
            "n_claims"           : n_claims,
            "supported_claims"   : supported,
            "faithfulness_score" : faithfulness,
            "avg_relevancy_score": relevancy,
            "retrieval_time_sec" : t_ret,
            "inference_time_sec" : t_inf,
            "overall_time_sec"   : overall,
        })
        _time.sleep(0.2)

ablation_detail_df = pd.DataFrame(ablation_rows)
ablation_summary   = (
    ablation_detail_df
    .groupby(["chunking","retrieval_variant"], as_index=False)
    .agg(
        queries           =("question","count"),
        faithfulness      =("faithfulness_score","mean"),
        relevancy         =("avg_relevancy_score","mean"),
        avg_retrieval_sec =("retrieval_time_sec","mean"),
        p95_retrieval_sec =("retrieval_time_sec", lambda s: s.quantile(0.95)),
        avg_inference_sec =("inference_time_sec","mean"),
        avg_overall_sec   =("overall_time_sec","mean"),
        p95_overall_sec   =("overall_time_sec", lambda s: s.quantile(0.95)),
    )
    .sort_values(["faithfulness","relevancy","avg_overall_sec"],
                 ascending=[False,False,True])
    .reset_index(drop=True)
)
ablation_summary["throughput_qps"] = 1.0 / ablation_summary["avg_overall_sec"].clip(lower=1e-9)

print(f"Ablation — {len(abl_qs)} questions × {len(ABLATION_VARIATIONS)} variants = {len(ablation_rows)} rows")
display(ablation_summary)


## 16. Save Outputs

In [ ]:
def resolve_output_dir() -> Path:
    for candidate in [Path("/kaggle/working")/"outputs", Path.cwd()/"outputs"]:
        try:
            candidate.mkdir(parents=True, exist_ok=True)
            return candidate
        except OSError:
            continue
    raise OSError("Could not create a writable output directory.")

out = resolve_output_dir()

_saves = {
    "evaluation_question_bank.csv": locals().get("eval_df"),
    "retrieval_metrics.csv"       : locals().get("results_df"),
    "qa_winner_comparison.csv"    : locals().get("qa_df"),
    "qa_artifacts.csv"            : locals().get("qa_arts_df"),
    "llm_judge_metrics.csv"       : locals().get("judge_df"),
    "llm_judge_examples.csv"      : locals().get("judge_examples"),
    "ablation_detail.csv"         : locals().get("ablation_detail_df"),
    "ablation_summary.csv"        : locals().get("ablation_summary"),
}

for fname, df_var in _saves.items():
    if df_var is not None and not df_var.empty:
        df_var.to_csv(out / fname, index=False)
        print(f"  Saved: {fname}  ({len(df_var)} rows)")
    else:
        print(f"  Skipped (not computed): {fname}")

print(f"\nOutputs directory: {out}")
